# 02 — Feature Engineering & Data Preparation

## Objective
Create credit-behaviour features, investigate categorical codes, prepare train/test datasets, and persist processed data for modelling.

**Leakage control:** ID is excluded from predictors. All learned preprocessing (imputation/scaling/encoding) will be fitted inside model pipelines using training data only.

In [1]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 1. Load data and engineer behavioural features

In [2]:
from src.data_preprocessing import load_credit_data, split_features_target
from src.feature_engineering import engineer_features
df = load_credit_data(PROJECT_ROOT / "data" / "raw")
df_fe = engineer_features(df)
print("Original shape:", df.shape, "| Engineered shape:", df_fe.shape)
display(df_fe.head())

Original shape: (30000, 25) | Engineered shape: (30000, 35)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT,MAX_REPAYMENT_STATUS,AVG_REPAYMENT_STATUS,COUNT_DELAYED_MONTHS,ANY_SEVERE_DELAY,AVG_BILL_AMOUNT,MAX_BILL_AMOUNT,CREDIT_UTILIZATION,AVG_PAYMENT_AMOUNT,TOTAL_PAYMENT_AMOUNT,PAYMENT_TO_BILL_RATIO
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1,2,-0.333333,2,1,1284.000000,3913,0.064200,114.833333,689,0.089434
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1,2,0.500000,2,1,2846.166667,3455,0.023718,833.333333,5000,0.292791
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0,0,0.000000,0,0,16942.166667,29239,0.188246,1836.333333,11018,0.108388
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0,0,0.000000,0,0,38555.666667,49291,0.771113,1398.000000,8388,0.036259
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0,0,-0.333333,0,0,18223.166667,35835,0.364463,9841.500000,59049,0.540054


## 2. Review categorical and ordinal codes

In [3]:
for col in ["SEX", "EDUCATION", "MARRIAGE", "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]:
    if col in df_fe:
        print(f"\n{col}:")
        print(df_fe[col].value_counts(dropna=False).sort_index())


SEX:
SEX
1    11888
2    18112
Name: count, dtype: int64

EDUCATION:
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

MARRIAGE:
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64

PAY_0:
PAY_0
-2     2759
-1     5686
 0    14737
 1     3688
 2     2667
 3      322
 4       76
 5       26
 6       11
 7        9
 8       19
Name: count, dtype: int64

PAY_2:
PAY_2
-2     3782
-1     6050
 0    15730
 1       28
 2     3927
 3      326
 4       99
 5       25
 6       12
 7       20
 8        1
Name: count, dtype: int64

PAY_3:
PAY_3
-2     4085
-1     5938
 0    15764
 1        4
 2     3819
 3      240
 4       76
 5       21
 6       23
 7       27
 8        3
Name: count, dtype: int64

PAY_4:
PAY_4
-2     4348
-1     5687
 0    16455
 1        2
 2     3159
 3      180
 4       69
 5       35
 6        5
 7       58
 8        2
Name: count, dtype: int64

PAY_5:
PAY_5
-2     4546
-1     5539
 

## 3. Review engineered feature distributions

In [4]:
engineered_cols = [c for c in ["MAX_REPAYMENT_STATUS", "AVG_REPAYMENT_STATUS",
    "COUNT_DELAYED_MONTHS", "ANY_SEVERE_DELAY", "AVG_BILL_AMOUNT",
    "MAX_BILL_AMOUNT", "CREDIT_UTILIZATION", "AVG_PAYMENT_AMOUNT",
    "TOTAL_PAYMENT_AMOUNT", "PAYMENT_TO_BILL_RATIO"] if c in df_fe]
display(df_fe[engineered_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
MAX_REPAYMENT_STATUS,30000.0,0.438733,1.345154,-2.000000,0.000000,0.000000,2.000000,8.000000e+00
AVG_REPAYMENT_STATUS,30000.0,-0.182439,0.982176,-2.000000,-0.833333,0.000000,0.000000,6.000000e+00
COUNT_DELAYED_MONTHS,30000.0,0.834200,1.554303,0.000000,0.000000,0.000000,1.000000,6.000000e+00
ANY_SEVERE_DELAY,30000.0,0.279333,0.448679,0.000000,0.000000,0.000000,1.000000,1.000000e+00
AVG_BILL_AMOUNT,30000.0,44976.945200,63260.721860,-56043.166667,4781.333333,21051.833333,57104.416667,8.773138e+05
MAX_BILL_AMOUNT,30000.0,60572.436867,78404.814025,-6029.000000,10060.000000,31208.500000,79599.000000,1.664089e+06
CREDIT_UTILIZATION,30000.0,0.373048,0.351890,-0.232590,0.029997,0.284834,0.687929,5.364308e+00
AVG_PAYMENT_AMOUNT,30000.0,5275.232094,10137.946323,0.000000,1113.291667,2397.166667,5583.916667,6.273443e+05
TOTAL_PAYMENT_AMOUNT,30000.0,31651.392567,60827.677939,0.000000,6679.750000,14383.000000,33503.500000,3.764066e+06
PAYMENT_TO_BILL_RATIO,29134.0,0.482380,5.926547,0.000000,0.042445,0.094831,0.616945,7.970000e+02


## 4. Missingness and outlier diagnostics

In [5]:
missing = df_fe.isna().mean().sort_values(ascending=False)
display((missing * 100).to_frame("missing_pct").head(20))
numeric_cols = df_fe.select_dtypes(include=np.number).columns.drop("DEFAULT", errors="ignore")
display(df_fe[numeric_cols].quantile([0, .01, .25, .5, .75, .99, 1]).T)

,missing_pct
PAYMENT_TO_BILL_RATIO,2.886667
AVG_REPAYMENT_STATUS,0.000000
PAY_AMT3,0.000000
PAY_AMT4,0.000000
PAY_AMT5,0.000000
PAY_AMT6,0.000000
DEFAULT,0.000000
MAX_REPAYMENT_STATUS,0.000000
COUNT_DELAYED_MONTHS,0.000000
PAY_AMT1,0.000000


,0.00,0.01,0.25,0.50,0.75,0.99,1.00
ID,1.000000,300.99,7500.750000,15000.500000,22500.250000,29700.010000,3.000000e+04
LIMIT_BAL,10000.000000,10000.00,50000.000000,140000.000000,240000.000000,500000.000000,1.000000e+06
SEX,1.000000,1.00,1.000000,2.000000,2.000000,2.000000,2.000000e+00
EDUCATION,0.000000,1.00,1.000000,2.000000,2.000000,5.000000,6.000000e+00
MARRIAGE,0.000000,1.00,1.000000,2.000000,2.000000,3.000000,3.000000e+00
AGE,21.000000,22.00,28.000000,34.000000,41.000000,60.000000,7.900000e+01
PAY_0,-2.000000,-2.00,-1.000000,0.000000,0.000000,3.000000,8.000000e+00
PAY_2,-2.000000,-2.00,-1.000000,0.000000,0.000000,3.000000,8.000000e+00
PAY_3,-2.000000,-2.00,-1.000000,0.000000,0.000000,3.000000,8.000000e+00
PAY_4,-2.000000,-2.00,-1.000000,0.000000,0.000000,3.000000,8.000000e+00


## 5. Split predictors and target

In [6]:
X_train, X_test, y_train, y_test = split_features_target(df_fe, test_size=0.20, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train default rate:", round(y_train.mean(), 4))
print("Test default rate:", round(y_test.mean(), 4))

Train: (24000, 33) Test: (6000, 33)
Train default rate: 0.2212
Test default rate: 0.2212


## 6. Persist processed datasets

In [7]:
processed = PROJECT_ROOT / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)
X_train.to_csv(processed / "X_train.csv", index=False)
X_test.to_csv(processed / "X_test.csv", index=False)
y_train.to_csv(processed / "y_train.csv", index=False)
y_test.to_csv(processed / "y_test.csv", index=False)
df_fe.to_csv(processed / "credit_risk_features.csv", index=False)
print("Processed datasets saved.")

Processed datasets saved.


## 7. Modelling considerations

- Repayment codes are ordinal-like but their exact semantics should be confirmed from the dataset documentation.
- The engineered ratios can be skewed or undefined when denominators are zero; missing values are left for pipeline imputation.
- Do not automatically cap or delete outliers without a business rationale.
- The random stratified split is a baseline only. This dataset does not provide a trustworthy application/performance date to construct a true out-of-time split.